In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect("../data/obras.db")

def sql(consulta):
    return pd.read_sql(consulta, con)

In [2]:
sql("""
SELECT idcontrato, preco_alterado
FROM fato_modificacao
ORDER BY preco_alterado DESC
LIMIT 5
""")

,idcontrato,preco_alterado
0,7682444,4.446039e+10
1,8547471,4.595660e+08
2,7682444,4.445567e+08
3,7392833,3.971910e+08
4,7392833,3.971910e+08


In [3]:
sql("SELECT name FROM sqlite_master WHERE type = 'table'")

,name
0,dim_entidade
1,dim_procedimento
2,dim_local
3,dim_cpv
4,fato_contrato
5,fato_modificacao


In [4]:
sql("PRAGMA table_info(fato_contrato)")

,cid,name,type,notnull,dflt_value,pk
0,0,idcontrato,INTEGER,0,None,1
1,1,id_entidade,INTEGER,0,None,0
2,2,id_procedimento,INTEGER,0,None,0
3,3,id_local,INTEGER,0,None,0
4,4,id_cpv,INTEGER,0,None,0
5,5,ano_celebracao,INTEGER,1,None,0
6,6,data_celebracao,TEXT,0,None,0
7,7,data_fecho,TEXT,0,None,0
8,8,tipo_fim,TEXT,0,None,0
9,9,preco_base,REAL,0,None,0


In [5]:
sql("SELECT * FROM fato_contrato LIMIT 3")

,idcontrato,id_entidade,id_procedimento,id_local,id_cpv,ano_celebracao,data_celebracao,data_fecho,tipo_fim,preco_base,...,atraso_dias_fecho,prorrogacao_dias,modificado,n_atos,causa_trab_complementares,causa_trab_a_menos,causa_revisao_ordinaria,causa_crise_precos,causa_interesse_publico,causa_prorrogacao
0,124837,2600,1,1,1,2012,2012-08-31,None,"O cumprimento, a impossibilidade definitiva e ...",295300.00,...,None,None,0,0,0,0,0,0,0,0
1,170263,2553,2,4,2,2012,2012-04-05,None,"O cumprimento, a impossibilidade definitiva e ...",149399.24,...,None,None,0,0,0,0,0,0,0,0
2,178493,1399,1,3,1,2012,2012-02-27,None,NaN,1750000.00,...,None,None,0,0,0,0,0,0,0,0


In [6]:
sql("""
SELECT idcontrato, ano_celebracao, preco_contratual, prazo_dias
FROM fato_contrato
LIMIT 5
""")

,idcontrato,ano_celebracao,preco_contratual,prazo_dias
0,124837,2012,268073.66,45
1,170263,2012,149351.24,90
2,178493,2012,1300000.00,550
3,178501,2012,671398.53,548
4,196934,2012,155085.67,90


In [7]:
sql("""
SELECT idcontrato, preco_contratual
FROM fato_contrato
WHERE ano_celebracao = 2021
ORDER BY preco_contratual DESC
LIMIT 10
""")

,idcontrato,preco_contratual
0,7632662,1.329000e+08
1,7622402,8.698926e+07
2,7569960,5.798500e+07
3,7773889,4.493267e+07
4,8465793,3.838834e+07
5,8426196,3.197000e+07
6,7724524,2.852852e+07
7,8146686,2.389929e+07
8,7806711,2.359522e+07
9,7662038,2.319507e+07


In [8]:
sql("""
SELECT idcontrato, preco_contratual, cpi_c
FROM fato_contrato
WHERE ano_celebracao = 2021 AND cpi_c IS NOT NULL
ORDER BY cpi_c
LIMIT 10
""")

,idcontrato,preco_contratual,cpi_c
0,7694469,8465962.29,0.525621
1,8962999,79700.00,0.548712
2,7787474,514571.70,0.580564
3,8231401,5478421.04,0.580695
4,8426196,31970000.00,0.603476
5,9212096,1644.33,0.622181
6,7986149,586420.32,0.624408
7,7588745,2849939.86,0.627472
8,7665986,263903.43,0.628520
9,7797068,499418.78,0.629338


In [9]:
sql(""" 
SELECT idcontrato, 
    preco_contratual, 
    cpi_c,
    ROUND((1/cpi_c - 1) *100, 2) AS estouro_pct 
FROM fato_contrato
WHERE ano_celebracao = 2021 AND cpi_c IS NOT NULL
ORDER BY cpi_c
LIMIT 10
""")

,idcontrato,preco_contratual,cpi_c,estouro_pct
0,7694469,8465962.29,0.525621,90.25
1,8962999,79700.00,0.548712,82.25
2,7787474,514571.70,0.580564,72.25
3,8231401,5478421.04,0.580695,72.21
4,8426196,31970000.00,0.603476,65.71
5,9212096,1644.33,0.622181,60.73
6,7986149,586420.32,0.624408,60.15
7,7588745,2849939.86,0.627472,59.37
8,7665986,263903.43,0.628520,59.10
9,7797068,499418.78,0.629338,58.90


In [10]:
sql("""
SELECT COUNT(*) AS total_contratos,
    COUNT(cpi_c) AS qtd_cpi_c,
    ROUND (AVG(cpi_c),3) AS media_cpi_c, 
    ROUND(AVG(preco_contratual),0) AS media_prec_contratual
FROM fato_contrato
WHERE ano_celebracao = 2021
""")

,total_contratos,qtd_cpi_c,media_cpi_c,media_prec_contratual
0,13844,6231,0.977,270263.0


In [11]:
sql("""
SELECT ano_celebracao,
    COUNT(*) AS total_contratos,
    COUNT(cpi_c) AS qtd_CPI_C,
    ROUND (AVG(cpi_c),3) AS media_CPI_C,
    SUM(cpi_c < 1) AS qtd_estouro,
    ROUND ((100.0 * SUM(cpi_c < 1) / COUNT(cpi_c)),1) AS percentual
FROM fato_contrato
WHERE ano_celebracao BETWEEN 2018 AND 2023
GROUP BY ano_celebracao
ORDER BY ano_celebracao
""")

,ano_celebracao,total_contratos,qtd_CPI_C,media_CPI_C,qtd_estouro,percentual
0,2018,10343,2814,1.005,622,22.1
1,2019,11521,4384,1.001,885,20.2
2,2020,12733,5591,0.994,1408,25.2
3,2021,13844,6231,0.977,2329,37.4
4,2022,9882,4174,0.982,1319,31.6
5,2023,12461,4486,0.985,1349,30.1


In [12]:
sql("PRAGMA table_info(dim_procedimento)")

,cid,name,type,notnull,dflt_value,pk
0,0,id_procedimento,INTEGER,0,None,1
1,1,tipo,TEXT,1,None,0


In [20]:
sql("""
SELECT f.idcontrato,
    f.preco_contratual,
    d.tipo
FROM fato_contrato AS f
JOIN dim_procedimento AS d
    ON f.id_procedimento = d.id_procedimento
WHERE f.ano_celebracao = 2021
ORDER BY preco_contratual DESC
LIMIT 5
""")

,idcontrato,preco_contratual,tipo
0,7632662,1.329000e+08,Concurso público
1,7622402,8.698926e+07,Concurso público
2,7569960,5.798500e+07,Concurso público
3,7773889,4.493267e+07,Concurso público
4,8465793,3.838834e+07,Concurso público


In [25]:
sql("""
SELECT d.tipo,
    COUNT(*) AS total_contratos,
    COUNT(cpi_c) AS qtd_cpi_c,
    ROUND ((100.0 * SUM(cpi_c < 1) / COUNT(cpi_c)),1) AS percentual
FROM fato_contrato AS f
JOIN dim_procedimento AS d
    ON f.id_procedimento = d.id_procedimento
WHERE f.ano_celebracao BETWEEN 2019 AND 2022
GROUP BY d.tipo
HAVING COUNT(cpi_c) >= 100
ORDER BY total_contratos DESC
""")

,tipo,total_contratos,qtd_cpi_c,percentual
0,Consulta Prévia,19845,8655,22.2
1,Concurso público,13886,5525,51.4
2,Ajuste Direto Regime Geral,12478,5157,15.0
3,Ao abrigo de acordo-quadro (art.º 258.º),1391,933,36.2
